# 🔬 STCL — Dual RP (Scan + Monitor separated)
**Scanning Transfer Cavity Lock** · RedPitaya STEMlab 125-14

Two RedPitayas with clearly separated responsibilities:

| Board | IP | Mode | Role |
|-------|----|------|------|
| `RP_Scan` | 192.168.0.201 | `scan` | Generates triangle ramp · locks cavity length via PID |
| `RP_Mon`  | 192.168.0.99  | `mon`  | Acquires cavity signal · runs live monitor · error monitor |

---
### 🔌 Wiring
```
RP_Scan  OUT2  ──→  piezo amp  ──→  FPI cavity input
RP_Scan  OUT2  ──→  RP_Mon IN2      (scan ramp as trigger reference)
FPI cavity output (photodiode) ──→  RP_Mon IN1  [HV mode, ±20 V]
RP_Mon   IN2   [HV mode, ±20 V]    (ramp ≤ 1 V — well within range)
```

---
### 📋 Execution order — run cells top to bottom, in sequence

| Phase | What happens | What you see |
|-------|-------------|--------------|
| 0 | Edit config | — |
| 1 | Connect boards, start event loop | Board IPs printed |
| 2 | Single acquisition sanity check | IN1 + IN2 plot |
| 3 | Push settings → start scan → open monitor | Qt window: peaks sweeping |
| 3b | *(optional)* Tune parameters live | Monitor updates |
| 4 | Start cavity lock — **monitor stays open** | Qt window: peak freezes at lockpoint |
| 5 | Switch to error monitor | Qt window: flat MHz trace = locked |
| 6 | *(optional)* Laser lock | — |
| 7 | Shutdown | — |

> **Key point:** Never call `stop_loop("RP_Scan")` manually between phases.
> `start_lock` stops the free-running scan internally before engaging the PID.
> The ramp output stays active throughout — the PID simply holds its DC offset.

---
## ⚙️ Phase 0: Configuration
<blockquote style="border-left:4px solid #e67e22; padding:6px 12px; background:#fdf6ec; color:#7f4f00; border-radius:4px;">
<strong>Edit only this section.</strong> All parameters flow through automatically.
</blockquote>

In [1]:
# ── Board IPs ─────────────────────────────────────────────────────────────────
RP_SCAN_IP = "192.168.0.201"   # Scan RP — generates ramp, runs cavity PID
RP_MON_IP  = "192.168.0.99"    # Monitor RP — acquires signal, displays monitor
SSH_USER   = "root"
SSH_PASS   = "root"

# ── Scan parameters (RP_Scan) ─────────────────────────────────────────────────
CAV_DEC    = 32      # decimation: 8 → 1.0 ms | 16 → 2.1 ms | 32 → 4.2 ms | 64 → 8.4 ms
CAV_AMP    = 0.4     # V — triangle ramp half-swing  (CAV_AMP + |CAV_OFFSET| ≤ 1.0 V)
CAV_OFFSET = 0.3     # V — DC offset on ramp

# ── Cavity lock parameters (RP_Scan PID) ──────────────────────────────────────
# CAV_RANGE: two time-windows [ms] bracketing the two reference peaks (FSR markers).
# Set these after looking at Phase 2 / Phase 3 — each window must contain exactly
# one clean peak. The PID drives the second peak to CAV_LOCKPOINT.
CAV_RANGE     = [[0.14, 0.30], [0.80, 0.90]] # ms — reference peak windows
CAV_LOCKPOINT = 0.86                         # ms — PID target (dashed line in monitor)                    
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Monitor display (RP_Mon) ───────────────────────────────────────────────────
# SHOW_TRIGGER = True  → dual-axis: IN1 (cavity transmission) + IN2 (scan ramp)
# SHOW_TRIGGER = False → single-axis: IN1 only
SHOW_TRIGGER = True

# ── Slave laser locks (optional — requires a third Lock RP) ───────────────────
# Uncomment and configure when adding a laser-lock RP.

# SL1_LABEL     = "Laser_A"
# SL1_RANGE     = [0.85, 1.10]   # ms
# SL1_LOCKPOINT = 0.96            # ms
# SL1_ENABLED   = True
# SL1_PID       = {"P": 0.0, "I": 0.5, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# SL2_LABEL     = "Laser_B"
# SL2_RANGE     = [0.50, 0.85]   # ms
# SL2_LOCKPOINT = 0.60            # ms
# SL2_ENABLED   = False
# SL2_PID       = {"P": 0.0, "I": 0.5, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

# ── Derived (do not edit) ─────────────────────────────────────────────────────
_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3
print(f"Scan period : {_CAV_PERIOD_MS:.3f} ms  (dec={CAV_DEC})")
print(f"Amp / Offset: {CAV_AMP} V / {CAV_OFFSET} V")
print(f"Cav range   : {CAV_RANGE}  lockpoint: {CAV_LOCKPOINT} ms")
print(f"Trigger view: {'ON — dual-axis (IN1 + IN2)' if SHOW_TRIGGER else 'OFF — IN1 only'}")

Scan period : 4.194 ms  (dec=32)
Amp / Offset: 0.4 V / 0.3 V
Cav range   : [[0.14, 0.3], [0.8, 0.9]]  lockpoint: 0.86 ms
Trigger view: ON — dual-axis (IN1 + IN2)


---
## 🔌 Phase 1: Import, Upload & Connect
Locates the repo, uploads RP-side scripts to both boards,
opens SSH connections, and starts the PC-side event loop.

> Run once per session. Re-running is safe — the event loop guard prevents duplicate threads.

In [2]:
# ── Imports & repo discovery ───────────────────────────────────────────────────
import sys, pathlib, threading, time
import numpy as np
import matplotlib.pyplot as plt

_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break
else:
    raise FileNotFoundError("lockclient.py not found — check repo location.")

from lockclient import LockClient, RP_client, Monitor

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL_dual_RP


In [3]:
# ── Build RP registry ─────────────────────────────────────────────────────────
Monitor.show_trigger = SHOW_TRIGGER

RPs = {
    "RP_Scan": RP_client((RP_SCAN_IP, 5000), {}, mode="scan"),
    "RP_Mon":  RP_client((RP_MON_IP,  5000), {}, mode="monitor"),
}

print("Uploading scripts and loading settings...")
Lock = LockClient(RPs)
print("Upload complete.")
for name, rp in Lock.RPs.items():
    print(f"  {name:8s}  mode={rp.mode:8s}  addr={rp.addr[0]}")

Uploading scripts and loading settings...
Upload complete.
  RP_Scan   mode=scan      addr=192.168.0.201
  RP_Mon    mode=monitor   addr=192.168.0.99


In [4]:
# ── SSH connect both boards ────────────────────────────────────────────────────
def _wrap(fn, err):
    try: fn()
    except Exception as exc: err["exc"] = exc

def run_with_timeout(fn, timeout_s, name):
    err = {}
    t = threading.Thread(target=lambda: _wrap(fn, err), daemon=True)
    t.start(); t.join(timeout=timeout_s)
    if t.is_alive():
        raise TimeoutError(f"{name} timed out after {timeout_s} s — check board SSH / network.")
    if "exc" in err:
        raise RuntimeError(f"{name} failed: {err['exc']}")

run_with_timeout(Lock.connect_all, timeout_s=45, name="connect_all")
print("Both boards connected.")

connecting...
Both boards connected.


In [5]:
# ── Start PC-side event loop + set decimation ─────────────────────────────────
if "stcl_thread" not in globals() or not stcl_thread.is_alive():
    stcl_thread = threading.Thread(target=Lock.start, daemon=True)
    stcl_thread.start()
    time.sleep(2)
    print("Event loop started.")
else:
    print("Event loop already running — skipping.")

Lock.set_dec("RP_Scan", CAV_DEC)
print(f"Decimation set: dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
print()
for name, rp in Lock.RPs.items():
    status = "connected" if rp.connected else "⚠ DISCONNECTED"
    print(f"  {name:8s}  {rp.addr[0]}  {status}")

Event loop started.
Decimation set: dec=32  period=4.194 ms

  RP_Scan   192.168.0.201  connected
  RP_Mon    192.168.0.99  connected


---
## 🔍 Phase 2: Signal Verification *(optional but recommended)*
Single acquisition from `RP_Mon` to confirm wiring before scanning.

**Expected:**
- `IN1` — cavity transmission peaks on a low baseline
- `IN2` — clean triangle ramp (directly from `RP_Scan OUT2`)

Use this to set `CAV_RANGE` in Phase 0 — the two peak windows must each
contain exactly one clearly visible transmission peak.

In [ ]:
acq = np.array(Lock.send("RP_Mon", "acquire"))

if acq.size == 0:
    print("⚠  No data — check RP_Mon is connected and IN1/IN2 are wired.")
else:
    t_ms, ch1, ch2 = acq[0], acq[1], acq[2]

    fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)

    axes[0].plot(t_ms, ch1, lw=0.8, color="#4fc3f7")
    axes[0].set_ylabel("IN1 [V]")
    axes[0].set_title("RP_Mon IN1 — Cavity transmission (HV mode)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(t_ms, ch2, lw=0.8, color="#a5d6a7")
    axes[1].set_ylabel("IN2 [V]")
    axes[1].set_title("RP_Mon IN2 — Scan ramp from RP_Scan OUT2 (trigger reference, HV mode)")
    axes[1].set_xlabel("Time [ms]")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"IN1 range: {ch1.min():.3f} … {ch1.max():.3f} V")
    print(f"IN2 range: {ch2.min():.3f} … {ch2.max():.3f} V")

---
## 📡 Phase 3: Scan + Live Monitor
Pushes settings to `RP_Scan`, starts the ramp on `OUT2`, and opens the
live monitor on `RP_Mon`.

**What you should see in the Qt window:**
- Two transmission peaks sweeping left and right as the ramp drives the cavity
- Coloured range-span markers at `CAV_RANGE` windows
- A dashed vertical line at `CAV_LOCKPOINT`

Tune `CAV_RANGE` and `CAV_LOCKPOINT` in Phase 3b until the dashed line sits
on the peak you want to lock to. Then proceed directly to Phase 4 — **do not
stop the scan or close the monitor before running Phase 4.**

In [6]:
# ── Push cavity settings to RP_Scan ───────────────────────────────────────────
Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("RP_Scan", "Master", "enabled",   True)
Lock.update_setting("RP_Scan", "Master", "PID",       CAV_PID)
print(f"Cavity settings pushed to RP_Scan")
print(f"  range={CAV_RANGE}  lockpoint={CAV_LOCKPOINT} ms")

check if lockpoint is still fine
Cavity settings pushed to RP_Scan
  range=[[0.14, 0.3], [0.8, 0.9]]  lockpoint=0.86 ms


In [7]:
# ── Start ramp on RP_Scan ─────────────────────────────────────────────────────
Lock.start_scan("RP_Scan", amplitude=CAV_AMP, offset=CAV_OFFSET)
print(f"Scan started on RP_Scan OUT2")
print(f"  amp={CAV_AMP} V  offset={CAV_OFFSET} V  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
time.sleep(3)   # allow scan + monitor server (port 5066) to initialise

Scan started on RP_Scan OUT2
  amp=0.4 V  offset=0.3 V  dec=32  period=4.194 ms
connected to <socket.socket fd=2444, family=2, type=1, proto=0, laddr=('192.168.0.137', 58355), raddr=('192.168.0.201', 5065)>


In [13]:
# ── Open live monitor on RP_Mon ───────────────────────────────────────────────
# Qt window opens. You should see transmission peaks sweeping with range markers.
Lock.start_monitor("RP_Mon")
print("Monitor open — peaks should be sweeping in the Qt window.")
print("Tune CAV_RANGE / CAV_LOCKPOINT in Phase 3b, then run Phase 4.")

Starting background process
monitoring process started
Monitor open — peaks should be sweeping in the Qt window.
Tune CAV_RANGE / CAV_LOCKPOINT in Phase 3b, then run Phase 4.


connected to <socket.socket fd=2740, family=2, type=1, proto=0, laddr=('192.168.0.137', 54014), raddr=('192.168.0.99', 5065)>


### 3b 🔧 Tune scan parameters live
Edit values and **re-run this cell** at any time while the scan is running.
The monitor updates immediately — no restart needed.

In [9]:
# ── Edit here, then re-run ────────────────────────────────────────────────────
CAV_DEC       = 32                            # scan period
CAV_AMP       = 0.4                           # V — ramp amplitude
CAV_OFFSET    = 0.3                           # V — ramp DC offset
CAV_RANGE     = [[0.14, 0.30], [0.80, 0.90]] # ms — reference peak windows
CAV_LOCKPOINT = 0.866                         # ms — PID target (dashed line in monitor) 

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

Lock.set_dec("RP_Scan", CAV_DEC)
if Lock.RPs["RP_Scan"].loop_running:
    Lock.set_scan_output("RP_Scan", amplitude=CAV_AMP, offset=CAV_OFFSET)
Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)

print(f"Updated — dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f} ms")
print(f"          amp={CAV_AMP} V  offset={CAV_OFFSET} V")
print(f"          range={CAV_RANGE}  lockpoint={CAV_LOCKPOINT} ms")

check if lockpoint is still fine
Updated — dec=32  period=4.194 ms
          amp=0.4 V  offset=0.3 V
          range=[[0.14, 0.3], [0.8, 0.9]]  lockpoint=0.866 ms


---
## 🔒 Phase 4: Cavity Lock

This cell stops the free-running scan internally, then engages the PID on
`RP_Scan`. The ramp output on `OUT2` stays active — the PID now holds its
DC offset to keep the cavity resonance pinned at `CAV_LOCKPOINT`.

**The monitor window stays open.** Watch it while this cell runs:
- The peak will initially sweep as normal
- Within a few scan cycles the PID will pull it toward the lockpoint
- Once locked, the peak sits **stationary at the dashed lockpoint line**

> This cell blocks until `Lock.stop_loop("RP_Scan")` is called (Phase 7
> or the shutdown cell). Run Phase 5 in a **separate cell** after this one
> has started — do not wait for it to return.

In [10]:
# ── Refresh settings and start cavity lock ────────────────────────────────────
# start_lock() stops the scan loop internally before engaging the PID.
# Do NOT call stop_loop() manually before this cell — it would kill the ramp.
Lock.update_setting("RP_Scan", "Master", "range",     CAV_RANGE)
Lock.update_setting("RP_Scan", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("RP_Scan", "Master", "enabled",   True)
Lock.update_setting("RP_Scan", "Master", "PID",       CAV_PID)
print(f"Starting cavity lock — target: {CAV_LOCKPOINT} ms")
print("Monitor window stays open: watch the peak stop sweeping and freeze at the lockpoint.")
print("Run Phase 5 in the next cell to switch to the error monitor.")
print()

Lock.start_lock("RP_Scan")   # blocks while lock is running

# The lines below execute only after the lock is stopped (Phase 7 / Lock.close())
print("Cavity lock stopped.")

check if lockpoint is still fine
Starting cavity lock — target: 0.866 ms
Monitor window stays open: watch the peak stop sweeping and freeze at the lockpoint.
Run Phase 5 in the next cell to switch to the error monitor.

Cavity lock stopped.


connected to <socket.socket fd=2320, family=2, type=1, proto=0, laddr=('192.168.0.137', 58667), raddr=('192.168.0.201', 5065)>


---
## 📊 Phase 5: Error Monitor
Run this cell **while Phase 4 is still blocking** (lock loop running).

Switches `RP_Mon` from the peak-position view to a frequency-error time trace
(MHz vs time). A flat line near 0 MHz means the cavity is locked.

**How to read it:**
- Flat, low noise around 0 MHz → cavity locked, PID holding
- Slow drift → PID integral gain too low, or lock losing
- Sudden jumps → peak escaped the range (cavity unlocked)

In [11]:
# ── Switch to error monitor ───────────────────────────────────────────────────
# Close the peak monitor first, then open the error monitor.
# Both run on RP_Mon — only one can be active at a time.
Lock.stop_monitor("RP_Mon")
time.sleep(0.5)

Lock.start_error_monitor("RP_Mon", tmin=20e-3)
print("Error monitor open on RP_Mon.")
print("Flat trace near 0 MHz = cavity locked.")
print("To stop error monitor: Lock.stop_monitor('RP_Mon')")

Starting background process
monitoring process started
Error monitor open on RP_Mon.
Flat trace near 0 MHz = cavity locked.
To stop error monitor: Lock.stop_monitor('RP_Mon')


In [12]:
Lock.stop_monitor('RP_Mon')

In [ ]:
# ── Save error trace to JSON (optional) ───────────────────────────────────────
# Uncomment and run to save the currently recorded error trace to disk.

# import time as _t
# filename = "lock_errors_{}".format(int(_t.time()))
# Lock.monitors["RP_Mon"]["queue_err"].put(("save", filename))
# print("Saved →", filename + ".json")

---
## 🔒 Phase 6: Laser Lock *(optional — requires a third Lock RP)*
Uncomment the slave config in Phase 0, add the Lock1 RP to the registry in
Phase 1, then run this cell while the cavity lock (Phase 4) is still running.

In [ ]:
# ── Push slave settings and start laser lock ───────────────────────────────────
if "Lock1" not in Lock.RPs:
    print("Lock1 not present — add RP_client for Lock1 in Phase 1, then re-run.")
else:
    Lock.update_setting("Lock1", "Slave1", "label",     SL1_LABEL)
    Lock.update_setting("Lock1", "Slave1", "range",     SL1_RANGE)
    Lock.update_setting("Lock1", "Slave1", "lockpoint", SL1_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave1", "enabled",   SL1_ENABLED)
    Lock.update_setting("Lock1", "Slave1", "PID",       SL1_PID)

    Lock.update_setting("Lock1", "Slave2", "label",     SL2_LABEL)
    Lock.update_setting("Lock1", "Slave2", "range",     SL2_RANGE)
    Lock.update_setting("Lock1", "Slave2", "lockpoint", SL2_LOCKPOINT)
    Lock.update_setting("Lock1", "Slave2", "enabled",   SL2_ENABLED)
    Lock.update_setting("Lock1", "Slave2", "PID",       SL2_PID)

    Lock.start_lock("Lock1")
    print(f"Laser lock started — Slave1: {SL1_LABEL}  Slave2: {SL2_LABEL}")
    print("To stop: Lock.stop_loop('Lock1')")

---
## 🛑 Phase 7: Safe Shutdown
`Lock.close()` stops everything in the correct order:
monitors → lock loops → scan → disconnect.

Run this when you are done. It will also unblock Phase 4 if still running.

In [ ]:
# ── Full automatic shutdown ───────────────────────────────────────────────────
Lock.close()
print("All monitors stopped, all loops halted, all boards disconnected.")

In [ ]:
# ── Manual step-by-step shutdown (if needed for debugging) ────────────────────
# Lock.stop_monitor("RP_Mon")
# time.sleep(0.5)
#
# if "Lock1" in Lock.RPs and Lock.RPs["Lock1"].loop_running:
#     Lock.stop_loop("Lock1")
#     time.sleep(0.5)
#
# if Lock.RPs["RP_Scan"].loop_running:
#     Lock.stop_loop("RP_Scan")
#     time.sleep(0.5)
#
# Lock.close()
# print("Manual shutdown complete.")